# Z-Score Anomaly Detection
A Z-score (or standard score) is a statistical measure that expresses how far a data point is from the mean of its distribution, measured in units of standard deviation. A positive Z-score means the value is above the mean, a negative Z-score means it is below, and a Z-score of zero equals the mean.It is a universal way to measure how unusual or typical a data point is within its distribution, making it essential for probability, hypothesis testing, and comparing across different datasets.

### Table of Contents
1. What is Z-Score?
2. Key Concepts
3. Why Modified Z-Score?
4. Implementation with `ifri_mini_ml_lib` 
5. Interactive Demo
6. Real-Life Applications
7. Limits and Challenges
8. Comparison between ifri_mini_ml_lib and scikit-learn
9. References

In [1]:
# Import required libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import warnings
warnings.filterwarnings('ignore')
import importlib.util
chemin_z_score = r"C:\Users\felic\Documents\ifri_mini_ml_lib\ifri_mini_ml_lib\anomalies_detection\z_score.py"


if os.path.exists(chemin_z_score):
    print(f"Fichier trouvé: {chemin_z_score}")
else:
    print(f" Fichier NON trouvé: {chemin_z_score}")
    print("   Vérifie le chemin")


spec = importlib.util.spec_from_file_location("z_score", chemin_z_score)
z_score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(z_score)

# Recuperation of the functions
modified_zscore_detection = z_score.modified_zscore_detection

# Scikit-learn imports
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

print("All libraries imported successfully!")
print(f"Custom Z-Score module loaded from: {os.path.abspath('..')}")


 Fichier NON trouvé: C:\Users\felic\Documents\ifri_mini_ml_lib\ifri_mini_ml_lib\anomalies_detection\z_score.py
   Vérifie le chemin


FileNotFoundError: [Errno 2] No such file or directory: '/home/rosas/Documents/IFRI/IFRI_COURS/2024_2025/Quatrième_Semestre/Concepts_&_Applications_du_ML/ifri_mini_ml_lib/notebooks/anomalies_detections/C:\\Users\\felic\\Documents\\ifri_mini_ml_lib\\ifri_mini_ml_lib\\anomalies_detection\\z_score.py'

## 1. What is Z-Score? 

### Definition
The **Z-Score** (also called **standard score**) is a statistical measurement that describes a value's relationship to the mean of a group of values.

### The Formula

$$ Z = \frac{X - \mu}{\sigma} $$

Where:
- **X** = The value to analyze
- **μ (mu)** = Mean of the dataset
- **σ (sigma)** = Standard deviation of the dataset

### Interpretation
| Z-Score | Meaning |
|---------|---------|
| Z = 0 | Value equals the mean |
| -1 < Z < 1 | Within 1 standard deviation (~68% of data) |
| -2 < Z < 2 | Within 2 standard deviations (~95% of data) |
| -3 < Z < 3 | Within 3 standard deviations (~99.7% of data) |
| \|Z\| > 3 | **ANOMALY** (outside 99.7% confidence interval) |

### Simple Example
If the average height of students is 170cm with a standard deviation of 10cm:
- A student of 190cm: Z = (190 - 170)/10 = **2.0** → Tall but normal
- A student of 210cm: Z = (210 - 170)/10 = **4.0** → **ANOMALY!** 

## 2. Key Concepts 

### 2.1 Normal Distribution (Gaussian)
The Z-Score assumes your data follows a **bell curve** (normal distribution).

```python
# Visual demonstration of normal distribution
x = np.linspace(-4, 4, 100)
y = 1/(np.sqrt(2*np.pi)) * np.exp(-x**2/2)

plt.figure(figsize=(10, 6))
plt.plot(x, y, 'b-', linewidth=2)
plt.fill_between(x, y, where=(x >= -3) & (x <= 3), alpha=0.3, color='green', label='Normal region (99.7%)')
plt.fill_between(x, y, where=(x < -3) | (x > 3), alpha=0.3, color='red', label='Anomaly region (0.3%)')
plt.axvline(x=-3, color='red', linestyle='--', alpha=0.7)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7)
plt.title('Normal Distribution with Anomaly Thresholds (|Z| > 3)')
plt.xlabel('Z-Score')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Why Modified Z-Score? 

### The Problem with Classic Z-Score

The classic Z-Score uses **mean** and **standard deviation**, which are **sensitive to outliers**!

### Demonstration

In [ ]:
# Create data with extreme outliers
np.random.seed(42)
normal_data = np.random.normal(100, 10, 90)  # 90 normal points
outliers = np.array([500, 600, 700, 800, 900, 1000])  # 6 extreme outliers
data_with_outliers = np.concatenate([normal_data, outliers])

print("=" * 60)
print("DEMONSTRATION: Classic Z-Score vs Modified Z-Score")
print("=" * 60)
print(f"Total data points: {len(data_with_outliers)}")
print(f"Normal points: 90")
print(f"Outliers added: 6")
print()

# Calculate statistics
classic_mean = np.mean(data_with_outliers)
classic_std = np.std(data_with_outliers)
robust_median = np.median(data_with_outliers)
mad = np.median(np.abs(data_with_outliers - robust_median))

print(f"CLASSIC STATISTICS (affected by outliers):")
print(f"   Mean: {classic_mean:.2f}")
print(f"   Standard Deviation: {classic_std:.2f}")
print()
print(f" ROBUST STATISTICS (resistant to outliers):")
print(f"   Median: {robust_median:.2f}")
print(f"   MAD (Median Absolute Deviation): {mad:.2f}")
print()

# Visual comparison
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(data_with_outliers, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(classic_mean, color='red', linewidth=2, label=f'Mean (classic): {classic_mean:.1f}')
plt.axvline(robust_median, color='green', linewidth=2, label=f'Median (robust): {robust_median:.1f}')
plt.title('Distribution with Outliers')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
# Box plot to show outliers clearly
plt.boxplot(data_with_outliers, vert=True)
plt.title('Box Plot Showing Outliers')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### The Modified Z-Score Formula

$$ M_i = \frac{0.6745 \times (x_i - \text{median})}{\text{MAD}} $$

Where:
- **MAD** = Median(|x_i - median|) = Median Absolute Deviation
- **0.6745** = Conversion factor to make MAD comparable to standard deviation

### Why is it better?

| Feature | Classic Z-Score | Modified Z-Score |
|---------|-----------------|------------------|
| Central tendency | Mean (sensitive) | Median (robust)  |
| Dispersion | Std Deviation (sensitive) | MAD (robust)  |
| Outlier resistance |  Low |  High |
| Recommended threshold | 3.0 | 3.5 |

### When to use Modified Z-Score?
- When your data already contains anomalies
- When you have a small dataset (< 30 points)
- When your distribution is not perfectly normal
- In production systems where robustness is critical

## 4. Implementation with `ifri_mini_ml_lib` 

### Creating a Sample Dataset
Let's create a realistic dataset: **Temperature readings from a sensor**

In [ ]:
# Create realistic temperature data
np.random.seed(42)

# Normal temperatures (20°C to 30°C)
normal_temps = np.random.normal(25, 2, 200)

# Add some anomalies (sensor failures, extreme events)
anomalies = np.array([15, 16, 40, 41, 42, 38, 39, 12, 13, 14])

# Combine
temperatures = np.concatenate([normal_temps, anomalies])
timestamps = np.arange(len(temperatures))

print("=" * 60)
print("DATASET: Sensor Temperature Readings")
print("=" * 60)
print(f"Total readings: {len(temperatures)}")
print(f"Normal readings: {len(normal_temps)}")
print(f"Anomalies injected: {len(anomalies)}")
print(f"Temperature range: {temperatures.min():.1f}°C - {temperatures.max():.1f}°C")
print(f"Mean temperature: {np.mean(temperatures):.1f}°C")

# Visualize the dataset
plt.figure(figsize=(14, 6))
plt.plot(timestamps, temperatures, 'o-', markersize=3, alpha=0.7)
plt.axhline(y=np.mean(temperatures), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(temperatures):.1f}°C')
plt.title('Sensor Temperature Readings (Red = Mean)')
plt.xlabel('Time (hours)')
plt.ylabel('Temperature (°C)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ANOMALY DETECTION with ifri_mini_ml_lib")
print("   Method: Modified Z-Score (Robust to outliers)")
print("=" * 60)

# Modified Z-Score detection
modified_anomalies, modified_zscores = modified_zscore_detection(
    temperatures, 
    threshold=3.5, 
    return_zscore=True
)

print(f"\n RESULTS:")
print(f"   • Total samples: {len(temperatures)}")
print(f"   • Anomalies detected: {sum(modified_anomalies)}")
print(f"   • Anomaly rate: {sum(modified_anomalies)/len(temperatures)*100:.2f}%")

# Anomaly details
anomaly_indices = np.where(modified_anomalies)[0].tolist()
if anomaly_indices:
    print(f"\n ANOMALY POSITIONS:")
    print(f"   {anomaly_indices}")

# Detection of injected anomalies
true_anomaly_positions = list(range(len(normal_temps), len(temperatures)))
detected = sum([1 for pos in true_anomaly_positions if modified_anomalies[pos]])

print(f"\n INJECTED ANOMALIES:")
print(f"   • Total injected: {len(anomalies)}")
print(f"   • Correctly detected: {detected}")
print(f"   • Detection rate: {detected/len(anomalies)*100:.0f}%")


In [ ]:
# Visualisation des anomalies détectées par Modified Z-Score
plt.figure(figsize=(14, 6))

colors = ['red' if a else 'blue' for a in modified_anomalies]
plt.scatter(timestamps, temperatures, c=colors, s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
plt.plot(timestamps, temperatures, 'gray', alpha=0.3, linewidth=1)

# Ligne de la médiane
median_val = np.median(temperatures)
plt.axhline(y=median_val, color='green', linestyle='--', linewidth=2, alpha=0.8, label=f'Median: {median_val:.1f}°C')

# Mise en forme
plt.title('Modified Z-Score Anomaly Detection (Red = Anomaly)', fontsize=14, fontweight='bold')
plt.xlabel('Time (hours)', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.grid(True, alpha=0.3, linestyle='--')

# Ajout des annotations pour les anomalies
anomaly_indices = np.where(modified_anomalies)[0]
for idx in anomaly_indices:
    plt.annotate('', xy=(idx, temperatures[idx]), xytext=(idx, temperatures[idx]+5),
                 arrowprops=dict(arrowstyle='->', color='red', alpha=0.7))

plt.tight_layout()
plt.show()

# Résumé
print("=" * 50)
print("MODIFIED Z-SCORE - SUMMARY")
print("=" * 50)
print(f"Total samples:        {len(temperatures)}")
print(f"Anomalies detected:   {sum(modified_anomalies)}")
print(f"Normal points:        {len(temperatures) - sum(modified_anomalies)}")
print(f"Detection rate:       {sum(modified_anomalies)/len(temperatures)*100:.2f}%")
print(f"Median temperature:   {median_val:.1f}°C")
print("=" * 50)

## 5. Interactive Demo 

### Play with the parameters and see real-time anomaly detection!

**Instructions:**
- Adjust the **threshold** to make detection more or less sensitive
- Change the **anomaly value** to see when it becomes an outlier
- The plot updates automatically!

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

@interact(
    threshold=FloatSlider(min=1.0, max=5.0, step=0.1, value=3.5, description='Threshold:'),
    anomaly_value=IntSlider(min=50, max=500, step=10, value=300, description='Anomaly value:')
)
def interactive_modified_demo(threshold=3.5, anomaly_value=300):
    """Interactive demo for Modified Z-Score anomaly detection"""
    
    # Create data with anomaly
    base_data = [100, 102, 98, 101, 99, 100, 101, 98, 102, 99, 100, 101]
    data_with_anomaly = base_data.copy()
    data_with_anomaly[5] = anomaly_value
    
    # Modified Z-Score detection
    anomalies, zscores = modified_zscore_detection(data_with_anomaly, threshold=threshold, return_zscore=True)
    
    # Results
    print(f"\n{'='*50}")
    print(f"MODIFIED Z-SCORE DEMO")
    print(f"{'='*50}")
    print(f"Test value: {anomaly_value}")
    print(f"Threshold: {threshold}")
    print(f"Modified Z-score: {zscores[5]:.2f}")
    print(f"Result: {' ANOMALY' if anomalies[5] else 'NORMAL'}")
    
    # Quick plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Data plot
    colors = ['red' if a else 'blue' for a in anomalies]
    ax1.scatter(range(len(data_with_anomaly)), data_with_anomaly, c=colors, s=80)
    ax1.axhline(y=np.median(data_with_anomaly), color='green', linestyle='--', label='Median')
    ax1.set_title('Data Points (Red = Anomaly)')
    ax1.set_xlabel('Index')
    ax1.set_ylabel('Value')
    ax1.legend()
    
    # Z-scores plot
    bar_colors = ['red' if abs(z) > threshold else 'blue' for z in zscores]
    ax2.bar(range(len(zscores)), zscores, color=bar_colors)
    ax2.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold (±{threshold})')
    ax2.axhline(y=-threshold, color='red', linestyle='--')
    ax2.set_title('Modified Z-Scores')
    ax2.set_xlabel('Index')
    ax2.set_ylabel('Modified Z-Score')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Real-Life Applications 

Z-Score anomaly detection is used across many industries:

| Industry | Application | Example |
|----------|-------------|---------|
|  **Finance** | Fraud detection | Unusually large transactions |
|  **IT/DevOps** | Server monitoring | CPU spikes, memory leaks |
|  **Manufacturing** | Quality control | Defective product dimensions |
|  **Automotive** | Predictive maintenance | Engine sensor anomalies |
|  **Healthcare** | Patient monitoring | Abnormal vital signs |
|  **Environment** | Climate monitoring | Extreme temperature events |
|  **Cybersecurity** | Intrusion detection | Unusual network traffic |
|  **Marketing** | Customer behavior | Unusual purchase patterns |

### Case Study: Credit Card Fraud Detection

In [ ]:
# Simulate credit card transactions
np.random.seed(42)

# Normal transactions: $20-$200
normal_transactions = np.random.normal(100, 40, 1000)
normal_transactions = np.clip(normal_transactions, 20, 500)

# Fraudulent transactions: $1000+
fraudulent = np.array([1500, 2300, 1800, 3200, 2900, 1100, 4500])

# Combine
all_transactions = np.concatenate([normal_transactions, fraudulent])

# Detect anomalies
anomalies, zscores = modified_zscore_detection(all_transactions, threshold=3.0, return_zscore=True)

print("=" * 60)
print("CASE STUDY: Credit Card Fraud Detection")
print("=" * 60)
print(f"Total transactions: {len(all_transactions)}")
print(f"Normal transactions: {len(normal_transactions)}")
print(f"Fraudulent transactions: {len(fraudulent)}")
print(f"Fraudulent transactions flagged: {sum(anomalies[-len(fraudulent):])}")

fraud_detection_rate = sum(anomalies[-len(fraudulent):]) / len(fraudulent) * 100
print(f"\n Fraud detection rate: {fraud_detection_rate:.0f}%")

# Show the transactions
fraud_indices = np.where(anomalies)[0]
print(f"\n Flagged suspicious transactions: {fraud_indices.tolist()}")
print(f"   Amounts: {[all_transactions[i] for i in fraud_indices]}")

## 7. Limits and Challenges 

### 1. Assumes Normal Distribution
Z-Score works best when data follows a bell curve. For non-normal distributions, it may produce false positives.

### 2. Sensitive to Multiple Anomalies
Classic Z-Score can be "blinded" by multiple anomalies (masking effect).

### 3. Batch Processing Only
Z-Score typically requires the entire dataset; not ideal for real-time streaming.

### 4. No Seasonality or Trends
Cannot handle:
- Cyclical patterns (e.g., higher traffic on weekends)
- Trends (e.g., gradual temperature increase)

### 5. Threshold Selection
Choosing the right threshold (2.5, 3.0, or 4.0) is context-dependent and impacts performance.

### When NOT to use Z-Score:

| Scenario | Alternative |
|----------|-------------|
| Time series with trend | ARIMA, Prophet |
| Seasonal data | Seasonal Z-Score |
| Non-normal distribution | IQR (Interquartile Range) |
| High-dimensional data | Isolation Forest, DBSCAN |
| Streaming data | Online anomaly detection |

### The Masking Effect Demonstration

In [ ]:
# Demonstrate the masking effect
np.random.seed(42)

# Clean data
clean_data = np.random.normal(100, 10, 100)
clean_anomalies = modified_zscore_detection(clean_data, threshold=3.0)

# Data with multiple anomalies
anomalies_list = [300, 310, 320, 330, 340, 350, 360, 370, 380, 390, 400]
corrupted_data = np.concatenate([clean_data, anomalies_list])
corrupted_anomalies = modified_zscore_detection(corrupted_data, threshold=3.0)

print("=" * 60)
print("THE MASKING EFFECT DEMONSTRATION")
print("=" * 60)
print(f"Clean data: {sum(clean_anomalies)} anomalies (ideal)")
print(f"Corrupted data (with 11 outliers): {sum(corrupted_anomalies)} anomalies detected")
print(f"Only {sum(corrupted_anomalies[-len(anomalies_list):])} out of {len(anomalies_list)} outliers identified!")
print("\nThe mean and standard deviation are skewed by the outliers,")
print("causing the anomalies to appear less extreme!")

# Visualize
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(clean_data, bins=30, edgecolor='black', alpha=0.7)
plt.title('Clean Data Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(corrupted_data, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(corrupted_data), color='red', linestyle='--', label='Skewed Mean')
plt.title('Corrupted Data Distribution')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

## 8. Comparison between ifri_mini_ml_lib and scikit-learn

In [ ]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import importlib

chemin_z_score=r"C:\Users\felic\Documents\ifri_mini_ml_lib\ifri_mini_ml_lib\anomalies_detection\z_score.py"
spec = importlib.util.spec_from_file_location("z_score", chemin_z_score)
z_score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(z_score)


modified_zscore_detection = z_score.modified_zscore_detection
# Import sklearn
from sklearn.covariance import EllipticEnvelope

# ============================================================================
# TEST DATA
# ============================================================================

np.random.seed(42)
data = np.concatenate([
    np.random.normal(100, 15, 950),  
    np.random.uniform(200, 500, 50)  
])
np.random.shuffle(data)

print(f" {len(data)} données, {50} reals anomalies")
print("=" * 60)

# ============================================================================
# TEST WITH ifri_mini_ml_lib
# ============================================================================

start = time.time()
anomalies_maison, scores = modified_zscore_detection(data, threshold=3.0, return_zscore=True)
time_maison = time.time() - start

print(f"\n ifri_mini_ml_lib:")
print(f"   Time: {time_maison*1000:.2f} ms")
print(f"   Anomalies: {sum(anomalies_maison)}")

# ============================================================================
# TEST WITH scikit-learn
# ============================================================================

start = time.time()
data_2d = data.reshape(-1, 1)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_2d)

elliptic = EllipticEnvelope(contamination=0.05, random_state=42)
predictions = elliptic.fit_predict(data_scaled)
anomalies_sklearn = predictions == -1
time_sklearn = time.time() - start

print(f"\n scikit-learn :")
print(f"   Time: {time_sklearn*1000:.2f} ms")
print(f"   Anomalies: {sum(anomalies_sklearn)}")

# ============================================================================
# COMPARISON
# ============================================================================

print("\n" + "=" * 60)
print("COMPARISON:")
print(f"   ifri_mini_ml_lib: {time_maison*1000:.2f} ms")
print(f"   scikit-learn:     {time_sklearn*1000:.2f} ms")





## 9. References 

### Academic References

1. **Aggarwal, C. C. (2017).**  
   *Outlier Analysis* (2nd ed.). Springer.  
   ISBN: 978-3-319-47578-3

2. **Iglewicz, B., & Hoaglin, D. C. (1993).**  
   *How to Detect and Handle Outliers*.  
   ASQC Quality Press. ISBN: 978-0873891680

3. **Grubbs, F. E. (1969).**  
   Procedures for Detecting Outlying Observations in Samples.  
   *Technometrics*, 11(1), 1-21.

### Online Resources

- [NIST/SEMATECH e-Handbook of Statistical Methods](https://www.itl.nist.gov/div898/handbook/)
- [Scikit-learn Outlier Detection Documentation](https://scikit-learn.org/stable/modules/outlier_detection.html)

### Software & Libraries

- `ifri_mini_ml_lib` - Custom implementation ([GitHub](https://github.com/IFRI-AI-Classes/ifri_mini_ml_lib))
- `scikit-learn` - Machine learning library
- `NumPy` - Numerical computing
- `Matplotlib` - Visualization
- `pandas` - Data manipulation

### About This Implementation

This notebook uses the custom `ifri_mini_ml_lib` library, which implements:
- `modified_zscore_detection()` - Robust Modified Z-Score with Median and MAD

---

## Thank You!